In [133]:
%load_ext autoreload
%autoreload 2

In [165]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
from data import CsvOHLCVLoader, MoexOHLCVLoader, BybitOHLCVLoader
from data.news import LentaNewsLoader, NewsDataFrame
from data.fundamentals import SmartLabFundamentalsLoader
from data.dataset_builder import build_dataset
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from backtest import VectorBTBacktester
from strategy import RSIReversalStrategy, BuyAndHoldStrategy, MACDTrendStrategy, EMATrendStrategy

from NLP.bert_functions import (
    compute_bert_sentiment,
    add_avg_sentiment,
    add_sentiment_std,
    add_pct_negative,
    add_sentiment_change,
)

from NLP.ner_functions import (
    compute_entity_mentions,
    add_entity_mentions,
    add_entity_share,
    add_total_entity_mentions,
    add_entity_groups_count,
    add_co_mention,
)

from NLP.topic_functions import (
    compute_topic_scores,
    add_topic_share,
    add_topic_intensity,
    add_topic_entropy,
    add_active_topics_count,
    add_dominant_topic,
    add_dominant_topic_id,
)



ImportError: cannot import name 'MACDTrendStrategy' from 'strategy' (/Users/georgijsozinov/VSprojects/NLP-impact-on-trading/strategy/__init__.py)

In [166]:
from strategy.rsi_reversal import RSIReversalStrategy
from strategy.buy_and_hold import BuyAndHoldStrategy
from strategy.macd_trend import MACDTrendStrategy
from strategy.ema_trend import EMATrendStrategy

**Загружаем OHLVC**

MoexLoader

In [75]:
df_moex = MoexOHLCVLoader(ticker="IMOEX", timeframe="1d", start="2025-01-01", end="2026-12-31",market= 'index',board = 'SNDX').load()
df_plzl = MoexOHLCVLoader(ticker="PLZL", timeframe="1d", start="2025-01-01", end="2026-12-31").load() 
df_sibn = MoexOHLCVLoader(ticker="SIBN", timeframe="1d", start="2025-01-01", end="2026-12-31").load()
df_sber = MoexOHLCVLoader(ticker="X5", timeframe="1d", start="2025-01-01", end="2026-12-31").load()

In [76]:
df_sber.head()

,open,high,low,close,volume
timestamp,,,,,
2025-01-08 21:00:00+00:00,2690.0,3000.0,2670.0,2803.0,6661057
2025-01-09 21:00:00+00:00,2800.0,2975.0,2783.0,2945.0,3454995
2025-01-12 21:00:00+00:00,2962.0,2999.5,2882.0,2905.0,2120539
2025-01-13 21:00:00+00:00,2920.0,2933.0,2884.0,2929.0,911219
2025-01-14 21:00:00+00:00,2936.5,2949.5,2891.0,2904.5,694614


CsvLoader

In [ ]:
df_moex = CsvOHLCVLoader(path="storage/IMOEX/2020-01-01-2025-12-31.csv", start="2020-01-01", end="2025-12-31") .load()['close']
df_plzl = CsvOHLCVLoader(path="storage/PLZL/2020-01-01-2025-12-31.csv", start="2020-01-01", end="2025-12-31").load()['close']
df_sibn = CsvOHLCVLoader(path="storage/SIBN/2020-01-01-2025-12-31.csv", start="2020-01-01", end="2025-12-31") .load()['close']
df_sber = CsvOHLCVLoader(path="storage/SBER/2020-01-01-2025-12-31.csv", start="2020-01-01", end="2025-12-31") .load()['close']

In [ ]:
df_sber.head()

**Загружаем новости**

С помощью LentaLoader

In [65]:
news = LentaNewsLoader(
    start_date="2025-01-01",
    end_date="2026-12-31",
    max_pages_per_day=1,
    save=True,
    filename="storage/lenta_all_news_2025-20026.csv",
).load()

Collected 13677 urls


Parsing Lenta: 100%|██████████| 13677/13677 [3:10:25<00:00,  1.20it/s]     


Saved 13673 rows to storage/lenta_all_news_2025-20026.csv


In [66]:
news.head()

,datetime,title,text,source,url
0,2024-06-25 00:05:00+00:00,28 июня: какой праздник отмечают в России и мире,Фото: Gonzalo Fuentes / Reuters 28 июня профес...,Lenta.ru,https://lenta.ru/news/2024/06/25/28-iyunya-kak...
1,2024-08-30 00:05:00+00:00,30 августа: какой праздник отмечают в России и...,Фото: Максим Богодвид / РИА Новости 30 августа...,Lenta.ru,https://lenta.ru/news/2024/08/30/30-avgusta-ka...
2,2024-11-15 00:05:00+00:00,15 ноября: какой праздник отмечают в России и ...,Фото: beeboys / Shutterstock / Fotodom 15 нояб...,Lenta.ru,https://lenta.ru/news/2024/11/15/15-noyabrya-k...
3,2024-12-31 00:03:00+00:00,Протестующие в Тбилиси накрыли праздничный стол,Фото: Daro Sulakauri / Reuters Протестующие на...,Lenta.ru,https://lenta.ru/news/2024/12/31/protestuyusch...
4,2024-12-31 00:23:00+00:00,Моряки с танкера Eagle S попали под следствие ...,Фото: Jussi Nukari / Reuters Семь моряков с та...,Lenta.ru,https://lenta.ru/news/2024/12/31/sem-moryakov-...


Или читаем из CSV, чтобы не ждать 100 лет каждый раз

In [74]:
news = NewsDataFrame.read_csv('storage/lenta_all_news_2025-20026.csv')
news

,datetime,title,text,source,url
0,2024-06-25 00:05:00+00:00,28 июня: какой праздник отмечают в России и мире,Фото: Gonzalo Fuentes / Reuters 28 июня профес...,Lenta.ru,https://lenta.ru/news/2024/06/25/28-iyunya-kak...
1,2024-08-30 00:05:00+00:00,30 августа: какой праздник отмечают в России и...,Фото: Максим Богодвид / РИА Новости 30 августа...,Lenta.ru,https://lenta.ru/news/2024/08/30/30-avgusta-ka...
2,2024-11-15 00:05:00+00:00,15 ноября: какой праздник отмечают в России и ...,Фото: beeboys / Shutterstock / Fotodom 15 нояб...,Lenta.ru,https://lenta.ru/news/2024/11/15/15-noyabrya-k...
3,2024-12-31 00:03:00+00:00,Протестующие в Тбилиси накрыли праздничный стол,Фото: Daro Sulakauri / Reuters Протестующие на...,Lenta.ru,https://lenta.ru/news/2024/12/31/protestuyusch...
4,2024-12-31 00:23:00+00:00,Моряки с танкера Eagle S попали под следствие ...,Фото: Jussi Nukari / Reuters Семь моряков с та...,Lenta.ru,https://lenta.ru/news/2024/12/31/sem-moryakov-...
...,...,...,...,...,...
13668,2026-06-24 02:31:00+00:00,Женщина испугалась лобковых вшей и столкнулась...,Фото: Daria Voronchuk / Shutterstock / Fotodom...,Lenta.ru,https://lenta.ru/news/2026/06/24/zhenschina-is...
13669,2026-06-24 02:32:00+00:00,В Европарламенте заговорили о мрачной реальнос...,Фото: Johanna Geron / Reuters Санкционная стра...,Lenta.ru,https://lenta.ru/news/2026/06/24/v-evroparlame...
13670,2026-06-24 02:34:00+00:00,Ким Чен Ын заявил об оснащении флота КНДР ядер...,Фото: North Korea's Korean Central News Agency...,Lenta.ru,https://lenta.ru/news/2026/06/24/kim-chen-yn-z...
13671,2026-06-24 22:00:00+00:00,Трамп поглумился над трансгендерами в женском ...,Фото: Evelyn Hockstein / Reuters Президент Сое...,Lenta.ru,https://lenta.ru/news/2026/06/24/tramp-poglumi...


Отфильтруем новости по тикеру и отрасли

*Полюс*

In [67]:
plzl_news = news.keywords(["PLZL", "Полюс", "акции полюса","акции Полюса",'золото','золотодобывающая','Сухой Лог','Золото'])
plzl_news.save('storage/PLZL.csv')

Saved 153 rows to storage/PLZL.csv


*Сбер*

In [68]:
sber_news = news.keywords(["SBER", "Сбер", "акции сбербанка","акции сбера",'банк','банки','банкоматы','GigaChat','gigachat','Gigachat'])
sber_news.save('storage/SBER.csv')

Saved 400 rows to storage/SBER.csv


*Сибнефть*

In [69]:
sibn_news = news.keywords(["SIBN", "Сибнефть", "акции сибнефти","акции сибинтек",'нефть','нефтепереработка','нефтяная','Газпром нефть','газпром нефть'])
sibn_news.save('storage/SIBN.csv')

Saved 136 rows to storage/SIBN.csv


*Моекс (индекс)*

In [70]:
imoex_news = news.keywords(["IMOEX", "Moex", "индекс Мосбиржи","индекс мосбиржи",'Моекс','моекс','имоекс','мосбиржи',' фондовый индекс'])
imoex_news.save('storage/IMOEX.csv')

Saved 2 rows to storage/IMOEX.csv


**Загружаем фундаменталы**

In [78]:
fundamentals = SmartLabFundamentalsLoader(
    tickers=["SBER", "PLZL", "SIBN"],
    metrics=[
        "p_e",
        "roe",
        "net_income"
    ],
    start="2025-01-01",
    end="2026-12-31",
    period_type="quarter",
    report_type="MSFO",
    save=True,
    filename="storage/fundamentals_quarterly.csv",
).load()

fundamentals


Ticker: SBER
  p_e: 5 rows
  roe: 5 rows
  net_income: 5 rows

Ticker: PLZL
  p_e: 5 rows
  roe: 5 rows
  net_income: 5 rows

Ticker: SIBN
  p_e: 5 rows
  roe: 5 rows
  net_income: 5 rows
Saved 16 rows to storage/fundamentals_quarterly.csv


,ticker,period,period_date,available_date,pe,roe,net_income
0,PLZL,2024Q4,2024-12-31,2025-01-31,NaN,241.1,175.4
1,PLZL,2025Q1,2025-03-31,2025-04-30,8.29,NaN,NaN
2,PLZL,2025Q2,2025-06-30,2025-07-31,7.16,149.8,172.5
3,PLZL,2025Q3,2025-09-30,2025-10-31,9.36,NaN,NaN
4,PLZL,2025Q4,2025-12-31,2026-01-31,10.40,115.6,141.7
5,PLZL,2026Q1,2026-03-31,2026-04-30,9.11,NaN,NaN
6,SBER,2025Q1,2025-03-31,2025-04-30,4.33,24.4,436.0
7,SBER,2025Q2,2025-06-30,2025-07-31,4.41,23.0,422.6
8,SBER,2025Q3,2025-09-30,2025-10-31,3.91,23.7,449.8
9,SBER,2025Q4,2025-12-31,2026-01-31,3.97,19.9,398.6


**Мерджим таблицы**

In [79]:
items = {
    "plzl": (df_plzl, plzl_news, "PLZL"),
    "sber": (df_sber, sber_news, "SBER"),
    "sibn": (df_sibn, sibn_news, "SIBN"),
}

final = {}

for name, (ohlcv_df, news_df, ticker) in items.items():
    final[name] = build_dataset(
        ohlcv=ohlcv_df,
        news=news_df,
        fundamentals=fundamentals,
        ticker=ticker,
        drop_nans=True,
    )

plzl_final = final["plzl"]
sber_final = final["sber"]
sibn_final = final["sibn"]

In [80]:
sber_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   timestamp       381 non-null    datetime64[ns, UTC]
 1   open            381 non-null    float64            
 2   high            381 non-null    float64            
 3   low             381 non-null    float64            
 4   close           381 non-null    float64            
 5   volume          381 non-null    int64              
 6   date            381 non-null    object             
 7   news            381 non-null    object             
 8   period          381 non-null    object             
 9   period_date     381 non-null    datetime64[ns]     
 10  available_date  381 non-null    datetime64[ns, UTC]
 11  pe              381 non-null    float64            
 12  roe             381 non-null    float64            
 13  net_income      381 non-null    flo

In [11]:
sber_final.head()

,timestamp,open,high,low,close,volume,date,news,period,period_date,available_date,pe,roe,net_income,ticker
0,2025-05-01 21:00:00+00:00,3341.0,3348.5,3265.5,3283.5,309149,2025-05-01,[Фото: Maksim Konstantinov / Globallookpress.c...,2025Q1,2025-03-31,2025-04-30 00:00:00+00:00,4.33,24.4,436.0,SBER
1,2025-05-02 21:00:00+00:00,3283.5,3306.5,3281.0,3300.0,43620,2025-05-02,[С начала 2025 года рубль сильнее других валют...,2025Q1,2025-03-31,2025-04-30 00:00:00+00:00,4.33,24.4,436.0,SBER
2,2025-05-03 21:00:00+00:00,3300.0,3368.0,3299.0,3315.0,79137,2025-05-03,[],2025Q1,2025-03-31,2025-04-30 00:00:00+00:00,4.33,24.4,436.0,SBER
3,2025-05-04 21:00:00+00:00,3321.5,3334.5,3200.0,3237.0,848443,2025-05-04,[Фото: Алексей Никольский / РИА Новости Глобал...,2025Q1,2025-03-31,2025-04-30 00:00:00+00:00,4.33,24.4,436.0,SBER
4,2025-05-05 21:00:00+00:00,3239.5,3266.5,3201.5,3238.5,575476,2025-05-05,[],2025Q1,2025-03-31,2025-04-30 00:00:00+00:00,4.33,24.4,436.0,SBER


In [81]:
plzl_final[["timestamp", "ticker", "period", "available_date", "pe", "roe", "net_income"]].head(20)

,timestamp,ticker,period,available_date,pe,roe,net_income
0,2025-02-02 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
1,2025-02-03 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
2,2025-02-04 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
3,2025-02-05 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
4,2025-02-06 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
5,2025-02-09 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
6,2025-02-10 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
7,2025-02-11 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
8,2025-02-12 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4
9,2025-02-13 21:00:00+00:00,PLZL,2024Q4,2025-01-31 00:00:00+00:00,NaN,241.1,175.4


**Все, все данные есть. Теперь начинаем исследование. В дальнейшей работе я беру датасет СБЕРА, потому что у сбера больше новостей 265**

PS: Но естественно архитектура построена таким образом, что можно взять абсолютно любой тикер, будь то тикер с моекса или с байбита

Оценим каждую новость по сантименту и сделаем из этого фактор: 1 - положительная; 0 - отрицательная

In [82]:
sber_final = compute_bert_sentiment(
    sber_final,
    news_col="news",
    batch_size=16,
)

sber_final = add_avg_sentiment(sber_final, window=7)
sber_final = add_sentiment_std(sber_final, window=7)
sber_final = add_pct_negative(sber_final, window=7)
sber_final = add_sentiment_change(sber_final, window=7)

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

**Теперь применим NER к нашим новостям:**

PS: Считаю что это глупо и не должно входить в архитектуру решения, так как новости уже осортированы по конкретному тикеру и сектору, а находить
дополнительные сущности, которые будто невозможно интерпретировать странно и нерелевантно. Но раз такое задание есть, то постараемся максимально эффективно и экономически-обосновано посчитать некоторые факторы

In [83]:
sber_entities = {
    "regulator": [
        "цб",
        "банк россии",
        "центробанк",
        "ключевая ставка",
        "ставка",
        "набиуллина",
        "регулятор",
    ],

    "state": [
        "правительство",
        "минфин",
        "госдума",
        "государство",
        "казначейство",
        "минэкономразвития",
    ],

    "bank_sector": [
        "банк",
        "банки",
        "втб",
        "т-банк",
        "тинькофф",
        "альфа-банк",
        "газпромбанк",
        "мкб",
        "совкомбанк",
    ],

    "sanctions": [
        "санкции",
        "ограничения",
        "блокирующие санкции",
        "сша",
        "ес",
        "евросоюз",
        "ofac",
        "sdn",
    ],

    "macro": [
        "инфляция",
        "рубль",
        "доллар",
        "курс",
        "ввп",
        "рецессия",
        "экономика",
        "денежно-кредитная политика",
    ],

    "market": [
        "акции",
        "рынок",
        "мосбиржа",
        "индекс",
        "индекс мосбиржи",
        "инвесторы",
        "торги",
        "дивиденды",
    ],

    "business_results": [
        "прибыль",
        "выручка",
        "капитал",
        "рентабельность",
        "кредитный портфель",
        "чистая прибыль",
        "отчетность",
    ],
}

In [84]:
sber_final = compute_entity_mentions(
    sber_final,
    entity_groups=sber_entities,
    news_col="news",
)

sber_final = add_entity_mentions(sber_final, group="regulator", window=30)
sber_final = add_entity_mentions(sber_final, group="sanctions", window=30)
sber_final = add_entity_mentions(sber_final, group="bank_sector", window=30)
sber_final = add_entity_mentions(sber_final, group="business_results", window=30)

sber_final = add_entity_share(sber_final, group="regulator", window=30)
sber_final = add_entity_share(sber_final, group="sanctions", window=30)

sber_final = add_total_entity_mentions(sber_final, window=30)
sber_final = add_entity_groups_count(sber_final, window=30)

sber_final = add_co_mention(
    sber_final,
    group_a="regulator",
    group_b="market",
    window=30,
)

sber_final = add_co_mention(
    sber_final,
    group_a="sanctions",
    group_b="market",
    window=30,
)

sber_final = add_co_mention(
    sber_final,
    group_a="business_results",
    group_b="market",
    window=30,
)

In [85]:
sber_final.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume', 'date', 'news',
       'period', 'period_date', 'available_date', 'pe', 'roe', 'net_income',
       'ticker', 'has_news', 'news_count', 'bert_sentiment_score',
       'bert_is_positive', 'bert_is_negative', 'avg_sentiment_7d',
       'sentiment_std_7d', 'pct_negative_7d', 'sentiment_change_7d',
       'daily_regulator_mentions', 'daily_regulator_flag',
       'daily_state_mentions', 'daily_state_flag',
       'daily_bank_sector_mentions', 'daily_bank_sector_flag',
       'daily_sanctions_mentions', 'daily_sanctions_flag',
       'daily_macro_mentions', 'daily_macro_flag', 'daily_market_mentions',
       'daily_market_flag', 'daily_business_results_mentions',
       'daily_business_results_flag', 'daily_total_entity_mentions',
       'daily_entity_groups_count', 'regulator_mentions_30d',
       'sanctions_mentions_30d', 'bank_sector_mentions_30d',
       'business_results_mentions_30d', 'regulator_share_30d',
       'sanction

Новые колонки добавились, теперь посмотрим что там вообще

In [86]:
sber_final[['daily_regulator_mentions', 'daily_regulator_flag',
       'daily_state_mentions', 'daily_state_flag',
       'daily_bank_sector_mentions', 'daily_bank_sector_flag',
       'daily_sanctions_mentions', 'daily_sanctions_flag',
       'daily_macro_mentions', 'daily_macro_flag', 'daily_market_mentions',
       'daily_market_flag', 'daily_business_results_mentions',
       'daily_business_results_flag', 'daily_total_entity_mentions',
       'daily_entity_groups_count', 'regulator_mentions_30d',
       'sanctions_mentions_30d', 'bank_sector_mentions_30d',
       'business_results_mentions_30d', 'regulator_share_30d',
       'sanctions_share_30d', 'total_entity_mentions_30d',
       'entity_groups_count_30d', 'daily_regulator_market_co_mention',
       'regulator_market_co_mention_30d', 'daily_sanctions_market_co_mention',
       'sanctions_market_co_mention_30d',
       'daily_business_results_market_co_mention',
       'business_results_market_co_mention_30d']].tail(20)

,daily_regulator_mentions,daily_regulator_flag,daily_state_mentions,daily_state_flag,daily_bank_sector_mentions,daily_bank_sector_flag,daily_sanctions_mentions,daily_sanctions_flag,daily_macro_mentions,daily_macro_flag,...,regulator_share_30d,sanctions_share_30d,total_entity_mentions_30d,entity_groups_count_30d,daily_regulator_market_co_mention,regulator_market_co_mention_30d,daily_sanctions_market_co_mention,sanctions_market_co_mention_30d,daily_business_results_market_co_mention,business_results_market_co_mention_30d
361,0,0,0,0,0,0,0,0,0,0,...,0.000000,0.200000,27.0,0.566667,0,0.0,0,1.0,0,0.0
362,0,0,0,0,0,0,2,1,0,0,...,0.000000,0.166667,23.0,0.466667,0,0.0,0,0.0,0,0.0
363,0,0,0,0,0,0,0,0,0,0,...,0.000000,0.200000,25.0,0.500000,0,0.0,0,0.0,0,0.0
364,0,0,0,0,0,0,0,0,0,0,...,0.000000,0.200000,24.0,0.466667,0,0.0,0,0.0,0,0.0
365,0,0,0,0,0,0,1,1,0,0,...,0.000000,0.166667,22.0,0.400000,0,0.0,0,0.0,0,0.0
366,0,0,0,0,0,0,0,0,0,0,...,0.000000,0.166667,22.0,0.400000,0,0.0,0,0.0,0,0.0
367,0,0,0,0,0,0,0,0,0,0,...,0.000000,0.166667,22.0,0.400000,0,0.0,0,0.0,0,0.0
368,0,0,0,0,0,0,0,0,0,0,...,0.000000,0.133333,16.0,0.333333,0,0.0,0,0.0,0,0.0
369,0,0,0,0,0,0,0,0,0,0,...,0.000000,0.100000,15.0,0.300000,0,0.0,0,0.0,0,0.0
370,0,0,0,0,0,0,0,0,0,0,...,0.000000,0.100000,15.0,0.300000,0,0.0,0,0.0,0,0.0


Колонки не пустые, и результат нас устраивает

**Topic Modeling**

In [87]:
sber_topics = {
    "regulation": [
        "цб",
        "банк россии",
        "ключевая ставка",
        "ставка",
        "регулятор",
        "денежно-кредитная политика",
        "инфляция",
    ],

    "sanctions": [
        "санкции",
        "ограничения",
        "сша",
        "ес",
        "евросоюз",
        "ofac",
        "sdn",
    ],

    "market": [
        "акции",
        "рынок",
        "мосбиржа",
        "индекс",
        "инвесторы",
        "торги",
        "капитализация",
    ],

    "financial_results": [
        "прибыль",
        "выручка",
        "чистая прибыль",
        "отчетность",
        "капитал",
        "рентабельность",
        "дивиденды",
    ],

    "banking_business": [
        "кредит",
        "ипотека",
        "депозиты",
        "кредитный портфель",
        "розничный бизнес",
        "корпоративный бизнес",
        "комиссионные доходы",
    ],
}

In [88]:
sber_final = compute_topic_scores(
    sber_final,
    topic_groups=sber_topics,
    news_col="news",
)

sber_final = add_topic_share(sber_final, topic="regulation", window=30)
sber_final = add_topic_share(sber_final, topic="sanctions", window=30)
sber_final = add_topic_share(sber_final, topic="market", window=30)
sber_final = add_topic_share(sber_final, topic="financial_results", window=30)
sber_final = add_topic_share(sber_final, topic="banking_business", window=30)

sber_final = add_topic_entropy(sber_final, window=30)
sber_final = add_active_topics_count(sber_final, window=30)
sber_final = add_dominant_topic_id(sber_final, topic_groups=sber_topics, window=30)

In [89]:
sber_final.columns

Index(['timestamp', 'open', 'high', 'low', 'close', 'volume', 'date', 'news',
       'period', 'period_date', 'available_date', 'pe', 'roe', 'net_income',
       'ticker', 'has_news', 'news_count', 'bert_sentiment_score',
       'bert_is_positive', 'bert_is_negative', 'avg_sentiment_7d',
       'sentiment_std_7d', 'pct_negative_7d', 'sentiment_change_7d',
       'daily_regulator_mentions', 'daily_regulator_flag',
       'daily_state_mentions', 'daily_state_flag',
       'daily_bank_sector_mentions', 'daily_bank_sector_flag',
       'daily_sanctions_mentions', 'daily_sanctions_flag',
       'daily_macro_mentions', 'daily_macro_flag', 'daily_market_mentions',
       'daily_market_flag', 'daily_business_results_mentions',
       'daily_business_results_flag', 'daily_total_entity_mentions',
       'daily_entity_groups_count', 'regulator_mentions_30d',
       'sanctions_mentions_30d', 'bank_sector_mentions_30d',
       'business_results_mentions_30d', 'regulator_share_30d',
       'sanction

In [90]:
topic_features = [
    "topic_regulation_share_30d",
    "topic_sanctions_share_30d",
    "topic_market_share_30d",
    "topic_financial_results_share_30d",
    "topic_banking_business_share_30d",
    "topic_entropy_30d",
    "active_topics_count_30d",
    "dominant_topic_id_30d",
]

In [91]:
sber_final[topic_features]

,topic_regulation_share_30d,topic_sanctions_share_30d,topic_market_share_30d,topic_financial_results_share_30d,topic_banking_business_share_30d,topic_entropy_30d,active_topics_count_30d,dominant_topic_id_30d
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1.000000,0.000000,0.000000,0.0,0.000000,-0.0,1.000000,1
2,1.000000,0.000000,0.000000,0.0,0.000000,-0.0,1.000000,1
3,1.000000,0.000000,0.000000,0.0,0.000000,0.0,0.666667,1
4,1.000000,0.000000,0.000000,0.0,0.000000,0.0,0.500000,1
...,...,...,...,...,...,...,...,...
376,0.333333,0.333333,0.222222,0.0,0.111111,0.0,0.166667,1
377,0.300000,0.400000,0.200000,0.0,0.100000,-0.0,0.200000,2
378,0.300000,0.400000,0.200000,0.0,0.100000,0.0,0.200000,2
379,0.272727,0.363636,0.272727,0.0,0.090909,-0.0,0.233333,2


Видим какое-то количество пустых Nan и нулей из-за того что показатли строятся на предыдущем окне. Но далее мы все это дело почистим

In [92]:
sber_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 77 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   timestamp                                 381 non-null    datetime64[ns, UTC]
 1   open                                      381 non-null    float64            
 2   high                                      381 non-null    float64            
 3   low                                       381 non-null    float64            
 4   close                                     381 non-null    float64            
 5   volume                                    381 non-null    int64              
 6   date                                      381 non-null    object             
 7   news                                      381 non-null    object             
 8   period                                    381 non-null    ob

In [93]:
sber_final[['bert_sentiment_score','news']].head(20)

,bert_sentiment_score,news
0,-0.106467,[Фото: Maksim Konstantinov / Globallookpress.c...
1,0.422465,[С начала 2025 года рубль сильнее других валют...
2,NaN,[]
3,-0.174402,[Фото: Алексей Никольский / РИА Новости Глобал...
4,NaN,[]
5,NaN,[]
6,NaN,[]
7,NaN,[]
8,0.987947,[Фото: Caitlin Ochs / Reuters Концерт американ...
9,-0.007670,[Фото: Kay Nietfeld / Globallookpress.com На У...


In [94]:
sber_final.to_csv('sber_final.csv')

In [95]:
sber_final["has_news"].value_counts(dropna=False)

has_news
1    211
0    170
Name: count, dtype: int64

In [96]:
sber_final["news_count"].sum()

287

*Удаляем ненужные колонки перед регрессией и чистим датасет*

In [97]:
df = sber_final.copy()

In [98]:
regression_columns = [
    # базовые рыночные данные
    "timestamp",
    "open",
    "high",
    "low",
    "close",
    "volume",

    # фундаментальные факторы
    "pe",
    "roe",
    "net_income",

    # новостная активность
    "has_news",
    "news_count",

    # BERT
    "bert_sentiment_score",
    "bert_is_positive",
    "bert_is_negative",
    "avg_sentiment_7d",
    "sentiment_std_7d",
    "pct_negative_7d",
    "sentiment_change_7d",

    # NER rolling-факторы
    "regulator_mentions_30d",
    "sanctions_mentions_30d",
    "bank_sector_mentions_30d",
    "business_results_mentions_30d",
    "regulator_share_30d",
    "sanctions_share_30d",
    "total_entity_mentions_30d",
    "entity_groups_count_30d",
    "regulator_market_co_mention_30d",
    "sanctions_market_co_mention_30d",
    "business_results_market_co_mention_30d",

    # Topic Modeling rolling-факторы
    "topic_regulation_share_30d",
    "topic_sanctions_share_30d",
    "topic_market_share_30d",
    "topic_financial_results_share_30d",
    "topic_banking_business_share_30d",
    "topic_entropy_30d",
    "active_topics_count_30d",
    "dominant_topic_id_30d",
]

regression_columns = [col for col in regression_columns if col in sber_final.columns]

sber_regression_df = sber_final[regression_columns].copy()

sber_regression_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 37 columns):
 #   Column                                  Non-Null Count  Dtype              
---  ------                                  --------------  -----              
 0   timestamp                               381 non-null    datetime64[ns, UTC]
 1   open                                    381 non-null    float64            
 2   high                                    381 non-null    float64            
 3   low                                     381 non-null    float64            
 4   close                                   381 non-null    float64            
 5   volume                                  381 non-null    int64              
 6   pe                                      381 non-null    float64            
 7   roe                                     381 non-null    float64            
 8   net_income                              381 non-null    float64            
 9  

In [99]:
df = sber_regression_df.copy()

*Теперь пропуски, заполняем их так ->*

BERT NaN → нет новостей → 0, но has_news/news_count остаются как индикатор

NER rolling NaN → нет прошлой истории → 0

Topic rolling NaN → нет прошлой истории → 0

Фундаментальные NaN → медиана

Цены/доходности/target → не заполняем, а потом dropna

In [100]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
df = df.sort_values("timestamp").reset_index(drop=True)

bert_zero_cols = [
    "bert_sentiment_score",
    "bert_is_positive",
    "bert_is_negative",
    "avg_sentiment_7d",
    "sentiment_std_7d",
    "pct_negative_7d",
    "sentiment_change_7d",
]

ner_zero_cols = [
    "regulator_mentions_30d",
    "sanctions_mentions_30d",
    "bank_sector_mentions_30d",
    "business_results_mentions_30d",
    "regulator_share_30d",
    "sanctions_share_30d",
    "total_entity_mentions_30d",
    "entity_groups_count_30d",
    "regulator_market_co_mention_30d",
    "sanctions_market_co_mention_30d",
    "business_results_market_co_mention_30d",
]

topic_zero_cols = [
    "topic_regulation_share_30d",
    "topic_sanctions_share_30d",
    "topic_market_share_30d",
    "topic_financial_results_share_30d",
    "topic_banking_business_share_30d",
    "topic_entropy_30d",
    "active_topics_count_30d",
    "dominant_topic_id_30d",
]

fundamental_median_cols = [
    "pe",
    "roe",
    "net_income",
]

market_median_cols = [
    "volume",
]

In [101]:
zero_cols = bert_zero_cols + ner_zero_cols + topic_zero_cols
zero_cols = [col for col in zero_cols if col in df.columns]

median_cols = fundamental_median_cols + market_median_cols
median_cols = [col for col in median_cols if col in df.columns]

df[zero_cols] = df[zero_cols].fillna(0)

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 37 columns):
 #   Column                                  Non-Null Count  Dtype              
---  ------                                  --------------  -----              
 0   timestamp                               381 non-null    datetime64[ns, UTC]
 1   open                                    381 non-null    float64            
 2   high                                    381 non-null    float64            
 3   low                                     381 non-null    float64            
 4   close                                   381 non-null    float64            
 5   volume                                  381 non-null    int64              
 6   pe                                      381 non-null    float64            
 7   roe                                     381 non-null    float64            
 8   net_income                              381 non-null    float64            
 9  

**Ну а теперь можно строить объясняющую регрессию**

In [102]:
df["return"] = df["close"].pct_change()
df["future_return_1d"] = df["close"].pct_change().shift(-1)

df["return_lag1"] = df["return"].shift(1)
df["return_lag2"] = df["return"].shift(2)

df["volatility_7d"] = df["return"].rolling(7).std().shift(1)
df["volatility_30d"] = df["return"].rolling(30).std().shift(1)

In [103]:
df_reg = df.replace([np.inf, -np.inf], np.nan).dropna()
df_reg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 349 entries, 31 to 379
Data columns (total 43 columns):
 #   Column                                  Non-Null Count  Dtype              
---  ------                                  --------------  -----              
 0   timestamp                               349 non-null    datetime64[ns, UTC]
 1   open                                    349 non-null    float64            
 2   high                                    349 non-null    float64            
 3   low                                     349 non-null    float64            
 4   close                                   349 non-null    float64            
 5   volume                                  349 non-null    int64              
 6   pe                                      349 non-null    float64            
 7   roe                                     349 non-null    float64            
 8   net_income                              349 non-null    float64            
 9   has

In [104]:
target = "future_return_1d"

market_features = [
    "return_lag1",
    "return_lag2",
    "volatility_7d",
    "volatility_30d",
    "volume_change_lag1",
]

fundamental_features = [
    "pe",
    "roe",
    "net_income",
]

bert_features = [
    "has_news",
    "news_count",
    "bert_sentiment_score",
    "bert_is_positive",
    "bert_is_negative",
    "avg_sentiment_7d",
    "sentiment_std_7d",
    "pct_negative_7d",
    "sentiment_change_7d",
]

ner_features = [
    "regulator_mentions_30d",
    "sanctions_mentions_30d",
    "bank_sector_mentions_30d",
    "business_results_mentions_30d",
    "regulator_share_30d",
    "sanctions_share_30d",
    "total_entity_mentions_30d",
    "entity_groups_count_30d",
    "regulator_market_co_mention_30d",
    "sanctions_market_co_mention_30d",
    "business_results_market_co_mention_30d",
]

topic_features = [
    "topic_regulation_share_30d",
    "topic_sanctions_share_30d",
    "topic_market_share_30d",
    "topic_financial_results_share_30d",
    "topic_banking_business_share_30d",
    "topic_entropy_30d",
    "active_topics_count_30d",
    "dominant_topic_id_30d",
]

In [105]:
features_market = [col for col in market_features if col in df_reg.columns and df_reg[col].nunique() > 1]
features_fund = [col for col in market_features + fundamental_features if col in df_reg.columns and df_reg[col].nunique() > 1]
features_bert = [col for col in market_features + fundamental_features + bert_features if col in df_reg.columns and df_reg[col].nunique() > 1]
features_ner = [col for col in market_features + fundamental_features + bert_features + ner_features if col in df_reg.columns and df_reg[col].nunique() > 1]
features_full = [col for col in market_features + fundamental_features + bert_features + ner_features + topic_features if col in df_reg.columns and df_reg[col].nunique() > 1]

In [106]:
X_market = sm.add_constant(df_reg[features_market])
y_market = df_reg[target]
model_market = sm.OLS(y_market, X_market).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

X_fund = sm.add_constant(df_reg[features_fund])
y_fund = df_reg[target]
model_fund = sm.OLS(y_fund, X_fund).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

X_bert = sm.add_constant(df_reg[features_bert])
y_bert = df_reg[target]
model_bert = sm.OLS(y_bert, X_bert).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

X_ner = sm.add_constant(df_reg[features_ner])
y_ner = df_reg[target]
model_ner = sm.OLS(y_ner, X_ner).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

X_full = sm.add_constant(df_reg[features_full])
y_full = df_reg[target]
model_full = sm.OLS(y_full, X_full).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

In [107]:
regression_results = pd.DataFrame([
    {
        "model": "Market",
        "n_obs": int(model_market.nobs),
        "features": len(features_market),
        "r2": model_market.rsquared,
        "adj_r2": model_market.rsquared_adj,
        "significant_5pct": int((model_market.pvalues.drop("const", errors="ignore") < 0.05).sum()),
        "significant_10pct": int((model_market.pvalues.drop("const", errors="ignore") < 0.10).sum()),
    },
    {
        "model": "Market + Fundamentals",
        "n_obs": int(model_fund.nobs),
        "features": len(features_fund),
        "r2": model_fund.rsquared,
        "adj_r2": model_fund.rsquared_adj,
        "significant_5pct": int((model_fund.pvalues.drop("const", errors="ignore") < 0.05).sum()),
        "significant_10pct": int((model_fund.pvalues.drop("const", errors="ignore") < 0.10).sum()),
    },
    {
        "model": "Market + Fundamentals + BERT",
        "n_obs": int(model_bert.nobs),
        "features": len(features_bert),
        "r2": model_bert.rsquared,
        "adj_r2": model_bert.rsquared_adj,
        "significant_5pct": int((model_bert.pvalues.drop("const", errors="ignore") < 0.05).sum()),
        "significant_10pct": int((model_bert.pvalues.drop("const", errors="ignore") < 0.10).sum()),
    },
    {
        "model": "Market + Fundamentals + BERT + NER",
        "n_obs": int(model_ner.nobs),
        "features": len(features_ner),
        "r2": model_ner.rsquared,
        "adj_r2": model_ner.rsquared_adj,
        "significant_5pct": int((model_ner.pvalues.drop("const", errors="ignore") < 0.05).sum()),
        "significant_10pct": int((model_ner.pvalues.drop("const", errors="ignore") < 0.10).sum()),
    },
    {
        "model": "Full NLP",
        "n_obs": int(model_full.nobs),
        "features": len(features_full),
        "r2": model_full.rsquared,
        "adj_r2": model_full.rsquared_adj,
        "significant_5pct": int((model_full.pvalues.drop("const", errors="ignore") < 0.05).sum()),
        "significant_10pct": int((model_full.pvalues.drop("const", errors="ignore") < 0.10).sum()),
    },
])

regression_results

,model,n_obs,features,r2,adj_r2,significant_5pct,significant_10pct
0,Market,349,4,0.018742,0.007332,1,1
1,Market + Fundamentals,349,7,0.022877,0.002819,1,1
2,Market + Fundamentals + BERT,349,16,0.044680,-0.001359,2,2
3,Market + Fundamentals + BERT + NER,349,27,0.064540,-0.014143,2,7
4,Full NLP,349,35,0.073043,-0.027329,2,5


In [108]:
coef_full = pd.DataFrame({
    "factor": model_full.params.index,
    "coef": model_full.params.values,
    "p_value": model_full.pvalues.values,
    "t_stat": model_full.tvalues.values,
})

coef_full = coef_full[coef_full["factor"] != "const"]
coef_full = coef_full.sort_values("p_value")

coef_full.head(20)

,factor,coef,p_value,t_stat
15,pct_negative_7d,0.015985,0.014385,2.447512
25,regulator_market_co_mention_30d,-0.006425,0.023120,-2.271441
23,total_entity_mentions_30d,0.000974,0.073826,1.787689
1,return_lag1,0.085789,0.093951,1.674915
18,sanctions_mentions_30d,-0.001033,0.098801,-1.650696
17,regulator_mentions_30d,-0.001095,0.152326,-1.431365
13,avg_sentiment_7d,0.011473,0.157686,1.412896
27,business_results_market_co_mention_30d,0.011902,0.180733,1.338503
19,bank_sector_mentions_30d,-0.001209,0.198014,-1.287229
33,topic_entropy_30d,0.080147,0.272381,1.097596


In [109]:
nlp_features = bert_features + ner_features + topic_features

coef_full[
    coef_full["factor"].isin(nlp_features)
].sort_values("p_value")

,factor,coef,p_value,t_stat
15,pct_negative_7d,0.015985,0.014385,2.447512
25,regulator_market_co_mention_30d,-0.006425,0.023120,-2.271441
23,total_entity_mentions_30d,0.000974,0.073826,1.787689
18,sanctions_mentions_30d,-0.001033,0.098801,-1.650696
17,regulator_mentions_30d,-0.001095,0.152326,-1.431365
13,avg_sentiment_7d,0.011473,0.157686,1.412896
27,business_results_market_co_mention_30d,0.011902,0.180733,1.338503
19,bank_sector_mentions_30d,-0.001209,0.198014,-1.287229
33,topic_entropy_30d,0.080147,0.272381,1.097596
30,topic_market_share_30d,-0.020735,0.298950,-1.038688


**Выясняем что только regulator_market_co_mention_30d pct_negative_7d значимы. Попробуем построить трехфакторную модеуль только с ними**

In [113]:
three_factor_features = [
    "pct_negative_7d",
    "regulator_market_co_mention_30d"
]

X_three = sm.add_constant(df_reg[three_factor_features])
y_three = df_reg["future_return_1d"]

model_three = sm.OLS(
    y_three,
    X_three
).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

print(model_three.summary())

                            OLS Regression Results                            
Dep. Variable:       future_return_1d   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                 -0.004
Method:                 Least Squares   F-statistic:                    0.3615
Date:                Wed, 24 Jun 2026   Prob (F-statistic):              0.697
Time:                        21:28:44   Log-Likelihood:                 954.59
No. Observations:                 349   AIC:                            -1903.
Df Residuals:                     346   BIC:                            -1892.
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

В полной модели факторы работали вместе с рыночными, фундаментальными, BERT, NER и topic-переменными. Там коэффициенты оценивались при контроле остальных факторов. Когда мы оставили только три NLP-фактора, они потеряли значимость, потому что сами по себе объясняют очень небольшую часть будущей доходности.

Это нормальная ситуация: значимость фактора в полной модели не означает, что он будет значим в изолированной модели.

**Теперь строим бэктест состоящий из обычной RSI reversal стратегии, а далее добавляем модель логистической регрессии на НЛП, чтобы проверить на сколько она повышает/понижает доходность**

In [167]:
bt_df = df.copy()

bt_df["timestamp"] = pd.to_datetime(bt_df["timestamp"], errors="coerce", utc=True)
bt_df = bt_df.sort_values("timestamp").reset_index(drop=True)
bt_df = bt_df.set_index("timestamp")

In [168]:
delta = bt_df["close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss
bt_df["rsi_14"] = 100 - (100 / (1 + rs))

In [169]:
bt_df["future_return_1d"] = bt_df["close"].pct_change().shift(-1)

long_threshold = 0.002
short_threshold = -0.002

bt_df["ml_target"] = 0
bt_df.loc[bt_df["future_return_1d"] > long_threshold, "ml_target"] = 1
bt_df.loc[bt_df["future_return_1d"] < short_threshold, "ml_target"] = -1

In [170]:
ema_fast = bt_df["close"].ewm(span=12, adjust=False).mean()
ema_slow = bt_df["close"].ewm(span=26, adjust=False).mean()

bt_df["macd"] = ema_fast - ema_slow
bt_df["macd_signal"] = bt_df["macd"].ewm(span=9, adjust=False).mean()


In [171]:
# EMA для EMATrendStrategy
bt_df["ema_8"] = bt_df["close"].ewm(span=8, adjust=False).mean()
bt_df["ema_200"] = bt_df["close"].ewm(span=200, adjust=False).mean()

# RSI 50
delta = bt_df["close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(50).mean()
avg_loss = loss.rolling(50).mean()

rs = avg_gain / avg_loss
bt_df["rsi_50"] = 100 - (100 / (1 + rs))

bt_df[["close", "ema_8", "ema_200", "rsi_50"]].tail()

,close,ema_8,ema_200,rsi_50
timestamp,,,,
2026-06-17 21:00:00+00:00,2370.0,2391.117621,2534.942748,48.734177
2026-06-18 21:00:00+00:00,2331.0,2377.758150,2532.913467,47.661623
2026-06-21 21:00:00+00:00,2226.0,2344.034117,2529.859601,42.204629
2026-06-22 21:00:00+00:00,2257.0,2324.693202,2527.144580,45.646917
2026-06-23 21:00:00+00:00,2186.0,2293.872490,2523.750107,41.352113


In [172]:
ml_features = [
    "pct_negative_7d",
    "regulator_market_co_mention_30d"
]

ml_features = [
    col for col in ml_features
    if col in bt_df.columns and bt_df[col].nunique() > 1
]

ml_features

['pct_negative_7d', 'regulator_market_co_mention_30d']

In [173]:
train_start = "2025-05-01"
train_end = "2025-08-31"

test_start = "2025-09-01"
test_end = "2026-12-31"

train_df = bt_df.loc[train_start:train_end].copy()
test_df = bt_df.loc[test_start:test_end].copy()

In [174]:
train_ml = (
    train_df[ml_features + ["ml_target"]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

test_ml = (
    test_df[ml_features]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

X_train = train_ml[ml_features]
y_train = train_ml["ml_target"]

X_test = test_ml[ml_features]

y_train.value_counts()

ml_target
-1    45
 1    44
 0    24
Name: count, dtype: int64

In [175]:
logit_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

logit_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[-1, 0, 1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](2,)","['pct_negative_7d','regulator_market_co_mention_30d']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,2
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [176]:
test_df["ml_signal"] = 0
test_df.loc[X_test.index, "ml_signal"] = logit_model.predict(X_test)

test_df["ml_signal"].value_counts()

ml_signal
1    137
0    131
Name: count, dtype: int64

In [177]:
backtest_df = test_df.copy()

backtest_df = backtest_df.replace([np.inf, -np.inf], np.nan)
backtest_df = backtest_df.dropna(subset=["open", "high", "low", "close", "rsi_14"])

ml_filter = backtest_df["ml_signal"]

In [178]:
backtester = VectorBTBacktester(
    init_cash=10_000,
    fees=0.001,
    slippage=0.0008,
    freq="1d",
    shift_orders=True,
    logging=False,
)

In [179]:
rsi_strategy = RSIReversalStrategy(
    rsi_col="rsi_14",
    oversold=30,
    overbought=70,
    exit_level=50,
    allow_short=True,
)

In [180]:
macd_strategy = MACDTrendStrategy(
    macd_col="macd",
    signal_col="macd_signal",
    allow_short=True,
)

In [184]:
ema_strategy = EMATrendStrategy(
    fast_ema="ema_8",
    slow_ema="ema_200",
    rsi_col="rsi_50",
    long_rsi_min=50,
    short_rsi_max=50
)

In [181]:
comparison = backtester.run_comparison(
    df=backtest_df,
    strategy=rsi_strategy,
    strategy_name="RSI Reversal",
    include_buy_and_hold=True,
    include_base_strategy=True,
    ml_filters={
        "Logit_3_NLP": backtest_df["ml_signal"],
    },
    entry_mode="strict",
)

comparison

,strategy,init_cash,final_value,sharpe,sortino,max_drawdown_%,equity_pct_change,win_rate_%,trades
0,B&H,10000.0,7484.3784,-1.3761,-1.7660,27.7952,-25.1562,0.0000,1
1,RSI Reversal,10000.0,9203.8915,-0.4153,-0.7074,16.5994,-7.9611,54.5455,12
2,RSI Reversal&ML_Logit_3_NLP,10000.0,9516.5984,-0.3678,-0.5816,16.5994,-4.8340,33.3333,3


In [182]:
comparison2 = backtester.run_comparison(
    df=backtest_df,
    strategy=macd_strategy,
    strategy_name="MACD Trend",
    include_buy_and_hold=True,
    include_base_strategy=True,
    ml_filters={
        "Logit_3_NLP": backtest_df["ml_signal"],
    },
    entry_mode="strict",
)

comparison2

,strategy,init_cash,final_value,sharpe,sortino,max_drawdown_%,equity_pct_change,win_rate_%,trades
0,B&H,10000.0,7484.3784,-1.3761,-1.7660,27.7952,-25.1562,0.0000,1
1,MACD Trend,10000.0,11567.9829,0.9235,1.5413,15.2991,15.6798,41.1765,18
2,MACD Trend&ML_Logit_3_NLP,10000.0,9329.6104,-0.9894,-1.3319,9.4329,-6.7039,20.0000,5


In [185]:
comparison3 = backtester.run_comparison(
    df=backtest_df,
    strategy=ema_strategy,
    strategy_name="EMA Trend",
    include_buy_and_hold=True,
    include_base_strategy=True,
    ml_filters={
        "Logit_3_NLP": backtest_df["ml_signal"],
    },
    entry_mode="strict",
)

comparison3

/Users/georgijsozinov/VSprojects/NLP-impact-on-trading/strategy/ema_trend.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev_long_state = long_state.shift(1).fillna(False)
/Users/georgijsozinov/VSprojects/NLP-impact-on-trading/strategy/ema_trend.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prev_short_state = short_state.shift(1).fillna(False)
/Users/georgijsozinov/VSprojects/NLP-impact-on-trading/strategy/ema_trend.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future ver

,strategy,init_cash,final_value,sharpe,sortino,max_drawdown_%,equity_pct_change,win_rate_%,trades
0,B&H,10000.0,7484.3784,-1.3761,-1.7660,27.7952,-25.1562,0.0000,1
1,EMA Trend,10000.0,10773.4135,0.5760,0.7562,19.7785,7.7341,58.3333,13
2,EMA Trend&ML_Logit_3_NLP,10000.0,8940.6563,-1.2182,-1.2247,11.5714,-10.5934,0.0000,1


Исходя из результатов исследования NLP в данной задаче не является хорошим выбором для реализации торговой стратегии, но мы проверяли только на сбере. Также возможно в модель можно добавить рыночные данные, чтобы она вместе с NLP факторами давала больше объясняющей силы.